In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from feature_engine.encoding import CountFrequencyEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import shap
import mlflow.sklearn
import mlflow
import os
import joblib
import json

import config

In [ ]:
# define Mlflow env. settings
mlflow.set_tracking_uri(uri=config.mlflow_uri)
%env MLFLOW_TRACKING_URI= config.mlflow_db
%env mlflow_tracking_username = config.mlflow_un
%env mlflow_tracking_password = config.mlflow_pw
# mlflow.enable_system_metrics_logging()
# mlflow.autolog(log_models=True, log_datasets=True)

In [ ]:
def list_folders(path):
    """Lists all folders in the given directory."""

    folders = []
    for entry in os.listdir(path):
        if os.path.isdir(os.path.join(path, entry)):
            folders.append(entry)
    return folders

In [ ]:
mlflow.set_experiment("T0083 - Audit Review") 
with mlflow.start_run(run_name=f"Sklearn-IF Model", nested=True) as greatgrandparent_run:
    for customer in config.customerList: 
        for auditLog in config.auditList:  
            FilterTypeList = list_folders(f'{config.data_location}FS_audit_type_table_name/{auditLog}/') 
            for FilterType in FilterTypeList:
                with mlflow.start_run(run_name=f"Audit Type context-{auditLog}-{FilterType}-{customer}", nested=True) as grandparent_run:
                    filepath = Path(f'{config.data_location}FS_audit_type_table_name/{auditLog}/{FilterType}/FS_audit_type_table_name_{auditLog}_{FilterType}_{customer}.parquet')
                    if filepath.is_file()==True:
                        dataFrame = pd.read_parquet(filepath, engine='pyarrow')
                        dataFrame = dataFrame.astype(str)
                        features = dataFrame.columns.tolist()
                        categoricalColumns = dataFrame.select_dtypes(include=['object']).columns.tolist()
                        encoder = CountFrequencyEncoder(encoding_method='frequency', variables=categoricalColumns, ignore_format=True)
                        # Add the if statement to check if the DataFrame has at least one row
                        if len(dataFrame) > 1:
                            train, test = train_test_split(dataFrame, train_size=0.7, test_size=0.30)
                            enc_train = encoder.fit_transform(train) 
                            enc_test = encoder.fit_transform(test)
                                                        
                            # Define model parameters
                            numEst = 100
                            contamFloat = 0.005
                            #sampleSize = 256
                            random = np.random.RandomState(1234)

                            # Initialize the model
                            if_model = IsolationForest(random_state=random, n_estimators=numEst, contamination=contamFloat)

                            # Start an MLflow run
                            with mlflow.start_run(run_name=f"Model Evaluation", nested=True) as parent_run:
                                # Log model parameters and train
                                mlflow.log_param("n_estimators", numEst)
                                mlflow.log_param("contamination", contamFloat)
                                mlflow.log_param("random_state", random)
                                if_model.fit(enc_train)        
                                
                                #Export model
                                logModel = mlflow.sklearn.log_model(sk_model=if_model,artifact_path=f"Sklearn-IF_{auditLog}_{FilterType}_{customer}_audit_type_model",registered_model_name=f"Sklearn-IF_{auditLog}_{FilterType}_{customer}_audit_type_model")
                                dirPath = f"{config.model_location}{auditLog}"
                                Path(dirPath).mkdir(parents=True, exist_ok=True) 
                                joblib.dump(if_model,f"{dirPath}/Sklearn-IF_{auditLog}_{FilterType}_{customer}_audit_type_model.pkl")

                                # Predict anomalies and generate histogram
                                predicted = if_model.predict(enc_test)            
                                predictedScore = if_model.score_samples(enc_test)

                                # Combine the scores with the original data
                                predictedDF = pd.concat([pd.DataFrame(data=predicted,columns=['anomaly']),test],axis=1)
                                predictedDF = pd.concat([pd.DataFrame(data=predictedScore,columns=['anomaly_score']),predictedDF],axis=1)
                                predictedDF = predictedDF[predictedDF['anomaly'] > 0]

                                if predictedDF.empty:
                                    print(f'No anomalies for {auditLog} {FilterType} {customer}')        
                                else:
                                    #Export Dataset
                                    mlflow.log_table(predictedDF,f"{customer}_{FilterType}_testDF.json")                         
                                    
                                    with mlflow.start_run(run_name=f"Explain_model", nested=True) as child_run:
                                        #Train explaination model
                                        explainer = shap.TreeExplainer(if_model, feature_names=if_model.feature_names_in_) 
                                        shap_values = explainer.shap_values(enc_train)

                                        ## plot generation
                                        shapBarplot = plt.figure(figsize=(9, 3))
                                        shap.summary_plot(shap_values, features=explainer.feature_names , plot_type="bar", max_display=15)
                                        mlflow.log_figure(shapBarplot, f"{customer}_{FilterType}_GlobalBarPlot.png")  

                                        # Get the absolute mean SHAP values for feature importance
                                        mean_abs_shap_values = np.abs(shap_values).mean(axis=0)

                                        # Sort features based on their importance
                                        sorted_indices = np.argsort(mean_abs_shap_values)[::-1][:10]  # Get indices of top 10 features
                                        top_10_shap_values = shap_values[:, sorted_indices]
                                        top_10_feature_names = [explainer.feature_names[i] for i in sorted_indices]
                                        top_10_mean_shap_values = mean_abs_shap_values[sorted_indices]

                                        data = {}
                                        for i in range(len(top_10_feature_names)):
                                            data[top_10_feature_names[i]] = top_10_mean_shap_values[i]

                                        json_data = json.dumps(data)
                                        mlflow.log_table(data,f"Var_Imp_{auditLog}_{FilterType}_{customer}_audit_type_model.json")
                                        dirPath = f"{config.model_location}{auditLog}/varimp"
                                        Path(dirPath).mkdir(parents=True, exist_ok=True) 
                                        with open(f'{dirPath}/Var_Imp_{auditLog}_{FilterType}_{customer}_audit_type_model.json', 'w') as outfile:
                                            outfile.write(json_data) 

                                        #Export Explainer model
                                        dirPath = f"{config.model_location}{auditLog}/"
                                        Path(dirPath).mkdir(parents=True, exist_ok=True) 
                                        joblib.dump(explainer,f"{dirPath}/Explainer_Sklearn-IF_{auditLog}_{FilterType}_{customer}_audit_type_model.pkl")
                                        #issue logging explainer bug 
                                        #logExplainerModel = mlflow.shap.log_explainer(shap.TreeExplainer(if_model, feature_names=if_model.feature_names_in_) ,artifact_path=f"Explainer_Sklearn-IF_{auditLog}_{FilterType}_{customer}_audit_type_model",registered_model_name=f"Explainer_Sklearn-IF_{auditLog}_{FilterType}_{customer}_audit_type_model")
                                        
                        
                        else:
                            print("DataFrame is empty. Skipping train/test split and encoding.") 
                            # Handle the case where the DataFrame is empty 
                            # (e.g., print a warning, raise an exception, etc.)

                        
                    else:
                        print("Error - Feature store file not found. Check directory.")        
                    

In [ ]:
import mlflow
mlflow.end_run()